# Fase 1: Ingesta con PySpark en Azure Synapse
## Carga desde Raw → Calidad → Quarantine → Processed

**Dataset:** `laptop_scrap_data.csv`  
**Origen:** `abfss://raw@stgabritenreiroprueba.dfs.core.windows.net/`  
**Destinos:** `abfss://processed@...` (datos limpios) y `abfss://quarantine@...` (registros anómalos)

> **Nota:** En Synapse, la variable `spark` ya viene inyectada. No es necesario crear SparkSession.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Storage account (hardcoded default, override via pipeline parameters if needed)
STORAGE_ACCOUNT = 'stgabritenreiroprueba'

# Try to override with pipeline parameter
try:
    from notebookutils import mssparkutils
    param = mssparkutils.notebook.getParameter('storage_account')
    if param and param.strip():
        STORAGE_ACCOUNT = param.strip()
except Exception:
    pass

RAW_PATH = f'abfss://raw@{STORAGE_ACCOUNT}.dfs.core.windows.net/laptop_scrap_data.csv'
PROCESSED_PATH = f'abfss://processed@{STORAGE_ACCOUNT}.dfs.core.windows.net/'
QUARANTINE_PATH = f'abfss://quarantine@{STORAGE_ACCOUNT}.dfs.core.windows.net/'

assert RAW_PATH and PROCESSED_PATH and QUARANTINE_PATH, 'Paths cannot be empty'

print(f'Storage: {STORAGE_ACCOUNT}')
print(f'Raw: {RAW_PATH}')
print(f'Processed: {PROCESSED_PATH}')
print(f'Quarantine: {QUARANTINE_PATH}')
print(f'Spark version: {spark.version}')

---
## 1. Lectura del dataset RAW con schema explícito

In [ ]:
# Schema explícito para evitar inferSchema=true (más fiable en producción)
laptop_schema = StructType([
    StructField('Company', StringType(), True),
    StructField('TypeName', StringType(), True),
    StructField('Inches', DoubleType(), True),
    StructField('ScreenResolution', StringType(), True),
    StructField('Cpu', StringType(), True),
    StructField('Gpu', StringType(), True),
    StructField('OpSys', StringType(), True),
    StructField('TouchScreen', IntegerType(), True),
    StructField('Ips', IntegerType(), True),
    StructField('X_res', IntegerType(), True),
    StructField('Y_res', IntegerType(), True),
    StructField('ppi', DoubleType(), True),
    StructField('Dedicated_Gpu', IntegerType(), True),
    StructField('Ram_GB', IntegerType(), True),
    StructField('Weight_kg', DoubleType(), True),
    StructField('SSD', IntegerType(), True),
    StructField('HHD', IntegerType(), True),
    StructField('Storage_Type', StringType(), True),
    StructField('Total_Storage_GB', IntegerType(), True),
    StructField('Storage_Category', StringType(), True),
    StructField('Price', DoubleType(), True),
])

df_raw = spark.read \
    .option('header', 'true') \
    .option('quote', '\"') \
    .option('escape', '\"') \
    .option('treatEmptyValuesAsNulls', 'true') \
    .schema(laptop_schema) \
    .csv(RAW_PATH)

print(f'Filas cargadas: {df_raw.count():,}')
print(f'Columnas: {len(df_raw.columns)}')
print(f'Particiones: {df_raw.rdd.getNumPartitions()}')

In [ ]:
df_raw.printSchema()

In [ ]:
df_raw.show(5, truncate=False, vertical=True)

---
## 2. Diagnóstico rápido de calidad

In [ ]:
# Perfil de nulos por columna
null_profile = df_raw.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
])
print('Conteo de nulos por columna:')
null_profile.show(vertical=True)

In [ ]:
total_rows = df_raw.count()

null_counts = df_raw.select([
    (F.count(F.when(F.col(c).isNull(), c)) / total_rows * 100).alias(c)
    for c in df_raw.columns
])

print('% de nulos por columna:')
null_counts.show(vertical=True)

In [ ]:
# Duplicados exactos
dup_count = df_raw.count() - df_raw.distinct().count()
print(f'Filas duplicadas exactas: {dup_count}')

In [ ]:
# Estadísticas descriptivas de columnas numéricas
numeric_cols = [c for c, t in df_raw.dtypes if t in ('int', 'bigint', 'double', 'float')]
print('Columnas numéricas:', numeric_cols)

df_raw.select(numeric_cols).summary().show(vertical=True, truncate=False)

---
## 3. Aislamiento de registros anómalos → Quarantine

Criterios de anomalía:
- Filas con `Price` nulo o <= 0
- `Ram_GB` nulo o <= 0
- `Weight_kg` nulo o <= 0
- `Inches` nulo o <= 0

In [ ]:
anomaly_condition = (
    F.col('Price').isNull() | (F.col('Price') <= 0) |
    F.col('Ram_GB').isNull() | (F.col('Ram_GB') <= 0) |
    F.col('Weight_kg').isNull() | (F.col('Weight_kg') <= 0) |
    F.col('Inches').isNull() | (F.col('Inches') <= 0)
)

df_quarantine = df_raw.filter(anomaly_condition)
df_clean = df_raw.filter(~anomaly_condition)

print(f'Registros en Quarantine: {df_quarantine.count():,}')
print(f'Registros Clean: {df_clean.count():,}')

In [ ]:
# Inspeccionar qué se fue a quarantine
df_quarantine.show(20, truncate=False)

---
## 4. Estandarización de tipos de datos

Correcciones aplicadas a AMBOS branches (clean y quarantine):
- `HHD` → renombrar a `HDD` (typo del scraping)
- Casts explícitos para garantizar tipos correctos

In [ ]:
def standardize_columns(df):
    return (df
        .withColumnRenamed('HHD', 'HDD')
        .withColumn('Ram_GB', F.col('Ram_GB').cast('int'))
        .withColumn('Weight_kg', F.col('Weight_kg').cast('double'))
        .withColumn('Inches', F.col('Inches').cast('double'))
        .withColumn('TouchScreen', F.col('TouchScreen').cast('int'))
        .withColumn('Ips', F.col('Ips').cast('int'))
        .withColumn('Dedicated_Gpu', F.col('Dedicated_Gpu').cast('int'))
        .withColumn('Price', F.col('Price').cast('double'))
        .withColumn('SSD', F.col('SSD').cast('int'))
        .withColumn('Total_Storage_GB', F.col('Total_Storage_GB').cast('int'))
    )

df_clean = standardize_columns(df_clean)
df_quarantine = standardize_columns(df_quarantine)

print('Schema processed:')
df_clean.printSchema()

---
## 5. Escritura a Processed (Parquet) y Quarantine (Parquet)

In [ ]:
df_clean.write \
    .mode('overwrite') \
    .option('compression', 'snappy') \
    .parquet(PROCESSED_PATH)

df_quarantine.write \
    .mode('overwrite') \
    .option('compression', 'snappy') \
    .parquet(QUARANTINE_PATH)

print('✅ Datos escritos en processed/ y quarantine/')

In [ ]:
# Verificación de escritura
df_check_processed = spark.read.parquet(PROCESSED_PATH)
df_check_quarantine = spark.read.parquet(QUARANTINE_PATH)

print(f'Processed - Filas: {df_check_processed.count():,} | Columnas: {len(df_check_processed.columns)}')
print(f'Quarantine - Filas: {df_check_quarantine.count():,} | Columnas: {len(df_check_quarantine.columns)}')

print('\nColumnas en processed:', df_check_processed.columns)
df_check_processed.show(3, truncate=False, vertical=True)

In [ ]:
# Devolver conteo de filas en quarantine para la pipeline
quarantine_count = df_quarantine.count()
print(f'\n📤 Exit value: {quarantine_count} filas en quarantine')
mssparkutils.notebook.exit(quarantine_count)